# 07.02 | Targeted two-factor HRV comparisons

**Project:** Heart rate variability in older adults from southern Chile  
**Phase:** 07 — Two-factor HRV profiles  
**Notebook:** 02 — Predefined, interpretable conditional categorical comparisons

## Research objective

Estimate **unadjusted contrasts within two-factor profiles** using the accepted analytical dataset. The goal is to describe whether observed HRV distributions and their uncertainty differ among meaningful combinations of sex, age group, and blood-pressure categories. This notebook is **not** an interaction test, a covariate-adjusted analysis, a longitudinal comparison, or a systematic search across all 66 possible categorical cells.

Six scientifically motivated pairs are included:

1. Sex × age group: female-versus-male contrasts within age strata, plus the 70–79 versus 60–69 contrast within each sex.
2. Sex × systolic blood-pressure category: sex contrasts within BP strata and high-versus-low SBP within each sex.
3. Sex × diastolic blood-pressure category: analogous conditional comparisons for DBP.
4. Joint systolic × diastolic BP: four simple contrasts holding the other BP category fixed, plus a clearly labeled high–high versus low–low diagonal profile contrast.
5. Age group × systolic BP: SBP high-versus-low within each age group and a 70–79 versus 60–69 age contrast within each SBP category.
6. Age group × diastolic BP: the analogous comparison; the 80+ × DBP ≥90 stratum is structurally empty and is **reported, not invented or merged**.

**Analyzed outcomes:** SDNN, RMSSD, HF, LF and VLF are the five main HRV measures; Mean RR and Mean HR are separate *complementary* cardiac measures.

**Scope and safeguards:** keep every accepted record and every existing category. Sparsely represented subgroups remain visible with sample sizes and imprecise intervals; the empty cell has no effect estimate or p-value. Pairwise Mann–Whitney U compares rank distributions; Cliff's delta captures stochastic dominance; the difference of medians is a separate descriptive estimand. A difference between two p-values is **not** evidence of an interaction. Do not equate conditional comparisons with adjustment for other covariates.

The preceding 07.01 atlas provides complete descriptive coverage of all ten categorical pairs. The subsequent 07.03 notebook is reserved for continuous-factor profiles and synthesis.

## 0. Configuration and outcome definitions

Run from the project root (or a descendant), or use `INPUT_FILE_OVERRIDE` to specify the accepted `outputs/dataset_final.csv`. The default output directory belongs solely to this notebook. `N_BOOT=1500` matches the confidence-interval configuration used in stage 06; lower this value only for a temporary execution test. Figures are saved regardless of whether inline display is enabled.

In [1]:
from __future__ import annotations

from pathlib import Path
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from IPython.display import display

INPUT_FILE_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None
N_BOOT = 1500
SEED = 20261128
CONFIDENCE_LEVEL = 0.95
ALPHA = 0.05
PLOT_DPI = 210
SHOW_FIGURES = False

PRIMARY = {
    "t2m_pre_sdnn": "SDNN", "t2m_pre_rmssd": "RMSSD", "t2m_pre_hf": "HF",
    "t2m_pre_lf": "LF", "t2m_pre_vlf": "VLF",
}
COMPLEMENTARY = {"t2m_pre_mean_rr": "Mean RR", "t2m_pre_mean_hr": "Mean HR"}
MEASURES = {**PRIMARY, **COMPLEMENTARY}

FACTOR_LEVELS = {
    "age_group": ["60-69", "70-79", "80+"],
    "sex_category": ["Female", "Male"],
    "sbp_category": ["<140 mmHg", ">=140 mmHg"],
    "dbp_category": ["<90 mmHg", ">=90 mmHg"],
}
BLOCK_TITLES = {
    "sex_age": "Sex × age group",
    "sex_sbp": "Sex × systolic BP category",
    "sex_dbp": "Sex × diastolic BP category",
    "sbp_dbp": "Systolic × diastolic BP categories",
    "age_sbp": "Age group × systolic BP category",
    "age_dbp": "Age group × diastolic BP category",
}
BLOCK_ORDER = list(BLOCK_TITLES)

NAVY, MUTED, TEAL, CORAL, BLUE = "#19394C", "#627887", "#247E90", "#C76D52", "#4E87A1"
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.02)
plt.rcParams.update({
    "figure.facecolor": "white", "savefig.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#DAE4EA", "axes.labelcolor": NAVY, "text.color": NAVY,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": "#E7EDF1",
    "grid.alpha": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "figure.constrained_layout.use": False, "axes.titleweight": "semibold",
})
print("Python", platform.python_version(), "| SciPy", stats.__version__ if hasattr(stats, '__version__') else 'available')

Python 3.14.10 | SciPy available


## 1. Load the accepted dataset

No rows, source categories, pressure thresholds, outcomes, or original values are modified. A minimal required-column check only identifies an incorrect input file. The notebook never overwrites stage 05, 06 or 07.01 artifacts.

In [2]:
def find_project_input():
    if INPUT_FILE_OVERRIDE is not None:
        return Path(INPUT_FILE_OVERRIDE).expanduser().resolve()
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        candidate = folder / "outputs" / "dataset_final.csv"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("outputs/dataset_final.csv not found; set INPUT_FILE_OVERRIDE.")

input_file = find_project_input()
project_root = input_file.parent.parent if input_file.parent.name == "outputs" else input_file.parent
output_root = (Path(OUTPUT_ROOT_OVERRIDE).expanduser().resolve() if OUTPUT_ROOT_OVERRIDE
               else project_root / "eda_results" / "hrv_associations" / "07_02_targeted_two_factor_hrv_comparisons")
tables_dir = output_root / "tables"
fig_dir = output_root / "figures" / "multipanels"
tables_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(input_file)
required = [*FACTOR_LEVELS, *MEASURES]
missing = sorted(set(required) - set(data.columns))
if missing:
    raise KeyError(f"Missing required columns in accepted CSV: {missing}")

print("Source:", input_file)
print("Records (unchanged):", len(data))
for factor, levels in FACTOR_LEVELS.items():
    print(factor, data[factor].value_counts().reindex(levels, fill_value=0).to_dict())
print("Output directory:", output_root)


def save_table(frame, filename):
    target = tables_dir / filename
    frame.to_csv(target, index=False, float_format="%.12g")
    return target

Source: /home/david/Desktop/colabs/nunez_cristian_colabs/heart_variability_rate_profiles/outputs/dataset_final.csv
Records (unchanged): 402
age_group {'60-69': 184, '70-79': 190, '80+': 28}
sex_category {'Female': 294, 'Male': 108}
sbp_category {'<140 mmHg': 298, '>=140 mmHg': 104}
dbp_category {'<90 mmHg': 354, '>=90 mmHg': 48}
Output directory: /home/david/Desktop/colabs/nunez_cristian_colabs/heart_variability_rate_profiles/eda_results/hrv_associations/07_02_targeted_two_factor_hrv_comparisons


## 2. Prespecified comparison ledger

Each comparison is a **pair of two-factor cells** specified before seeing the target HRV measurements. Every row states the direction of the contrast: `comparison minus reference`. Positive Cliff's delta means larger values are more likely in the comparison group; positive median difference means a larger sample median in that group. The two effects can have different signs because they summarize different distributional properties.

This ledger is defined by the joint-cell support and physiological interpretability, **not by the HRV p-values of stage 06**. The comparison of 80+ female with male is retained despite its small male subgroup (n=6 in the accepted cohort), and the 80+ high-DBP contrast remains explicitly non-estimable when its cell is empty. The diagonal joint-BP contrast changes both categories simultaneously and is *not* a simple effect.

In [3]:
contrast_specs = []

def add_contrast(block, factor_a, factor_b, cmp_a, cmp_b, ref_a, ref_b,
                 short_label, scientific_question, role="conditional_simple"):
    contrast_specs.append({
        "block": block, "factor_a": factor_a, "factor_b": factor_b,
        "comparison_a": cmp_a, "comparison_b": cmp_b,
        "reference_a": ref_a, "reference_b": ref_b,
        "contrast_label": short_label,
        "scientific_question": scientific_question,
        "contrast_role": role,
    })

# 1. Sex × age: sex contrasts in all age bands; adjacent central age bands within each sex.
for age in FACTOR_LEVELS["age_group"]:
    add_contrast("sex_age", "sex_category", "age_group",
                 "Female", age, "Male", age,
                 f"Female − Male | age {age}", "Sex distribution difference at a fixed age group")
for sex in FACTOR_LEVELS["sex_category"]:
    add_contrast("sex_age", "sex_category", "age_group",
                 sex, "70-79", sex, "60-69",
                 f"70–79 − 60–69 | {sex}", "Age-band distribution difference within a fixed sex")

# 2. Sex × SBP: both directions of conditioning, without interpreting a difference of p-values as interaction.
for bp in FACTOR_LEVELS["sbp_category"]:
    add_contrast("sex_sbp", "sex_category", "sbp_category",
                 "Female", bp, "Male", bp,
                 f"Female − Male | SBP {bp.replace(' mmHg','')}", "Sex contrast within fixed SBP category")
for sex in FACTOR_LEVELS["sex_category"]:
    add_contrast("sex_sbp", "sex_category", "sbp_category",
                 sex, ">=140 mmHg", sex, "<140 mmHg",
                 f"SBP high − low | {sex}", "High-versus-low SBP within fixed sex")

# 3. Sex × DBP.
for bp in FACTOR_LEVELS["dbp_category"]:
    add_contrast("sex_dbp", "sex_category", "dbp_category",
                 "Female", bp, "Male", bp,
                 f"Female − Male | DBP {bp.replace(' mmHg','')}", "Sex contrast within fixed DBP category")
for sex in FACTOR_LEVELS["sex_category"]:
    add_contrast("sex_dbp", "sex_category", "dbp_category",
                 sex, ">=90 mmHg", sex, "<90 mmHg",
                 f"DBP high − low | {sex}", "High-versus-low DBP within fixed sex")

# 4. Joint BP: four grid edges (one BP fixed) and one labeled diagonal comparison.
for dbp in FACTOR_LEVELS["dbp_category"]:
    add_contrast("sbp_dbp", "sbp_category", "dbp_category",
                 ">=140 mmHg", dbp, "<140 mmHg", dbp,
                 f"SBP high − low | DBP {dbp.replace(' mmHg','')}",
                 "SBP simple contrast at fixed DBP category")
for sbp in FACTOR_LEVELS["sbp_category"]:
    add_contrast("sbp_dbp", "sbp_category", "dbp_category",
                 sbp, ">=90 mmHg", sbp, "<90 mmHg",
                 f"DBP high − low | SBP {sbp.replace(' mmHg','')}",
                 "DBP simple contrast at fixed SBP category")
add_contrast("sbp_dbp", "sbp_category", "dbp_category",
             ">=140 mmHg", ">=90 mmHg", "<140 mmHg", "<90 mmHg",
             "Both high − both low", "Diagonal profile contrast; both BP categories differ",
             role="joint_diagonal_descriptive")

# 5. Age × SBP. 80+ is included despite n=7 in its high-SBP cell.
for age in FACTOR_LEVELS["age_group"]:
    add_contrast("age_sbp", "age_group", "sbp_category",
                 age, ">=140 mmHg", age, "<140 mmHg",
                 f"SBP high − low | age {age}", "SBP difference within age group")
for bp in FACTOR_LEVELS["sbp_category"]:
    add_contrast("age_sbp", "age_group", "sbp_category",
                 "70-79", bp, "60-69", bp,
                 f"70–79 − 60–69 | SBP {bp.replace(' mmHg','')}", "Central-age-band difference within SBP category")

# 6. Age × DBP. The 80+ high-DBP cell is explicitly included in the ledger (n=0).
for age in FACTOR_LEVELS["age_group"]:
    add_contrast("age_dbp", "age_group", "dbp_category",
                 age, ">=90 mmHg", age, "<90 mmHg",
                 f"DBP high − low | age {age}", "DBP difference within age group")
for bp in FACTOR_LEVELS["dbp_category"]:
    add_contrast("age_dbp", "age_group", "dbp_category",
                 "70-79", bp, "60-69", bp,
                 f"70–79 − 60–69 | DBP {bp.replace(' mmHg','')}", "Central-age-band difference within DBP category")

planned = pd.DataFrame(contrast_specs)
planned.insert(0, "contrast_id", [f"C{i:02d}" for i in range(1, len(planned)+1)])

def pick_cell(frame, factor_a, factor_b, cat_a, cat_b):
    return frame.loc[(frame[factor_a] == cat_a) & (frame[factor_b] == cat_b)]

ledger_rows = []
for c in planned.itertuples(index=False):
    comp = pick_cell(data, c.factor_a, c.factor_b, c.comparison_a, c.comparison_b)
    ref  = pick_cell(data, c.factor_a, c.factor_b, c.reference_a, c.reference_b)
    na, nb = len(comp), len(ref)
    status = ("empty_comparison_or_reference" if min(na,nb)==0 else
              "small_group_n_below_10" if min(na,nb)<10 else
              "uneven_or_modest_group_n_below_20" if min(na,nb)<20 else
              "both_groups_n_at_least_20")
    ledger_rows.append({**c._asdict(), "n_comparison": na, "n_reference": nb,
                        "support_status": status, "estimable_cell_pair": min(na,nb)>0})
ledger = pd.DataFrame(ledger_rows)
save_table(ledger, "planned_two_factor_contrasts.csv")
display(ledger[["contrast_id","block","contrast_label","n_comparison","n_reference","support_status"]])
print("Preplanned contrasts:", len(ledger), "| With both groups occupied:", ledger.estimable_cell_pair.sum())

,contrast_id,block,contrast_label,n_comparison,n_reference,support_status
0,C01,sex_age,Female − Male | age 60-69,141,43,both_groups_n_at_least_20
1,C02,sex_age,Female − Male | age 70-79,131,59,both_groups_n_at_least_20
2,C03,sex_age,Female − Male | age 80+,22,6,small_group_n_below_10
3,C04,sex_age,70–79 − 60–69 | Female,131,141,both_groups_n_at_least_20
4,C05,sex_age,70–79 − 60–69 | Male,59,43,both_groups_n_at_least_20
5,C06,sex_sbp,Female − Male | SBP <140,221,77,both_groups_n_at_least_20
6,C07,sex_sbp,Female − Male | SBP >=140,73,31,both_groups_n_at_least_20
7,C08,sex_sbp,SBP high − low | Female,73,221,both_groups_n_at_least_20
8,C09,sex_sbp,SBP high − low | Male,31,77,both_groups_n_at_least_20
9,C10,sex_dbp,Female − Male | DBP <90,264,90,both_groups_n_at_least_20


Preplanned contrasts: 28 | With both groups occupied: 27


## 3. Effect-size estimators and stratified bootstrap

**Mann–Whitney U:** two-sided rank-distribution comparison (`method='asymptotic'` to handle ties consistently). The null should not be read as exclusively a difference in medians. **Cliff's delta** is the pairwise probability difference `P(X_comparison > X_reference) − P(X_comparison < X_reference)`, with ties contributing zero. **Median difference** equals the observed comparison-group median minus the reference-group median, in original measurement units. The bootstrap independently resamples **within each of the two observed cells** and produces percentile 95% CIs for both effect sizes. No transformed values are used in inference.

Every targeted contrast is exported, including non-significant and non-estimable ones. A nonempty group with n<10 is flagged but not dropped or declared intrinsically non-estimable. With n=0 in either group, no two-sample estimator is defined. The intervals assume independent records and do not account for possible within-project clustering.

In [4]:
boot_rng = np.random.default_rng(SEED)


def cliffs_delta(a, b):
    """Signed stochastic dominance; exact handling of tied values."""
    xa, xb = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if len(xa)==0 or len(xb)==0:
        return np.nan
    sb = np.sort(xb)
    lower = np.searchsorted(sb, xa, side="left")
    upper = np.searchsorted(sb, xa, side="right")
    return float((lower - (len(sb) - upper)).sum() / (len(xa) * len(sb)))


def bootstrap_two_group(a, b):
    na, nb = len(a), len(b)
    if na < 2 or nb < 2 or N_BOOT < 2:
        return np.nan, np.nan, np.nan, np.nan
    median_samples = np.empty(N_BOOT, dtype=float)
    delta_samples = np.empty(N_BOOT, dtype=float)
    for k in range(N_BOOT):
        aa = a[boot_rng.integers(0, na, size=na)]
        bb = b[boot_rng.integers(0, nb, size=nb)]
        median_samples[k] = np.median(aa) - np.median(bb)
        delta_samples[k] = cliffs_delta(aa, bb)
    tail = (1 - CONFIDENCE_LEVEL)/2
    lo_m, hi_m = np.quantile(median_samples, [tail, 1-tail])
    lo_d, hi_d = np.quantile(delta_samples, [tail, 1-tail])
    return float(lo_m), float(hi_m), float(lo_d), float(hi_d)


def descriptives(x):
    if len(x)==0:
        return {"mean":np.nan,"sd":np.nan,"q05":np.nan,"q25":np.nan,"median":np.nan,
                "q75":np.nan,"q95":np.nan,"iqr":np.nan}
    q = np.quantile(x,[.05,.25,.5,.75,.95])
    return {"mean":float(np.mean(x)),"sd":float(np.std(x,ddof=1)) if len(x)>1 else np.nan,
            "q05":float(q[0]),"q25":float(q[1]),"median":float(q[2]),
            "q75":float(q[3]),"q95":float(q[4]),"iqr":float(q[3]-q[1])}


def bh_fdr(pvalues):
    """Benjamini–Hochberg, preserving NaN for undefined p-values."""
    p = np.asarray(pvalues,dtype=float)
    result=np.full(len(p), np.nan, dtype=float)
    valid=np.flatnonzero(np.isfinite(p))
    if not len(valid):
        return result
    order=valid[np.argsort(p[valid])]
    k=len(order)
    sorted_q = p[order] * k / np.arange(1,k+1)
    sorted_q=np.minimum.accumulate(sorted_q[::-1])[::-1]
    result[order]=np.minimum(sorted_q,1)
    return result

## 4. Estimate all predefined two-factor comparisons

Every source measurement remains on its original scale. For each of the seven measures and each ledger contrast, the table exports sample sizes, per-group medians, Cliff's delta, median difference, bootstrap intervals, Mann–Whitney U and two-sided p-value. Separate reference-group descriptive distributions are retained for transparent inspection.

In [5]:
effect_rows = []
profile_rows = []
for c in ledger.itertuples(index=False):
    cmp_df = pick_cell(data, c.factor_a, c.factor_b, c.comparison_a, c.comparison_b)
    ref_df = pick_cell(data, c.factor_a, c.factor_b, c.reference_a, c.reference_b)
    for outcome, label in MEASURES.items():
        cmp_raw = pd.to_numeric(cmp_df[outcome],errors="coerce").to_numpy(dtype=float)
        ref_raw = pd.to_numeric(ref_df[outcome],errors="coerce").to_numpy(dtype=float)
        a, b = cmp_raw[np.isfinite(cmp_raw)], ref_raw[np.isfinite(ref_raw)]
        oa, ob = descriptives(a), descriptives(b)
        cls = "primary_hrv" if outcome in PRIMARY else "complementary_cardiac"
        for side,values,summ in [("comparison",a,oa),("reference",b,ob)]:
            profile_rows.append({"contrast_id":c.contrast_id,"block":c.block,"contrast_label":c.contrast_label,
                                 "side":side,"outcome":outcome,"outcome_label":label,
                                 "n_valid":len(values),**summ})
        if len(a)>0 and len(b)>0:
            u_stat,p_raw=stats.mannwhitneyu(a,b,alternative="two-sided",method="asymptotic")
            delta=cliffs_delta(a,b)
            median_diff=float(np.median(a)-np.median(b))
            m_lo,m_hi,d_lo,d_hi=bootstrap_two_group(a,b)
            status="estimated" if min(len(a),len(b))>=2 else "point_estimate_only_n_1"
        else:
            u_stat=p_raw=delta=median_diff=m_lo=m_hi=d_lo=d_hi=np.nan
            status="not_estimable_empty_cell"
        effect_rows.append({
            "contrast_id":c.contrast_id,"block":c.block,"block_label":BLOCK_TITLES[c.block],
            "factor_a":c.factor_a,"factor_b":c.factor_b,
            "contrast_label":c.contrast_label,"contrast_role":c.contrast_role,
            "comparison_a":c.comparison_a,"comparison_b":c.comparison_b,
            "reference_a":c.reference_a,"reference_b":c.reference_b,
            "outcome":outcome,"outcome_label":label,"family":cls,
            "n_comparison_cell":c.n_comparison,"n_reference_cell":c.n_reference,
            "n_comparison_valid":len(a),"n_reference_valid":len(b),
            "support_status":c.support_status,"estimation_status":status,
            "median_comparison":oa["median"],"median_reference":ob["median"],
            "median_difference_comparison_minus_reference":median_diff,
            "median_diff_ci_low":m_lo,"median_diff_ci_high":m_hi,
            "cliffs_delta_comparison_vs_reference":delta,
            "cliffs_ci_low":d_lo,"cliffs_ci_high":d_hi,
            "mann_whitney_u":float(u_stat),"p_raw":float(p_raw),
        })

effects = pd.DataFrame(effect_rows)
comparison_profiles = pd.DataFrame(profile_rows)
save_table(comparison_profiles,"two_factor_comparison_group_profiles.csv")
print("Outcomes × planned contrasts:", len(effects))
print("No-estimate rows:",(effects.estimation_status=="not_estimable_empty_cell").sum())
display(effects[["block","contrast_label","outcome_label","n_comparison_valid",
                 "n_reference_valid","cliffs_delta_comparison_vs_reference","p_raw"]].head(14))

Outcomes × planned contrasts: 196
No-estimate rows: 7


,block,contrast_label,outcome_label,n_comparison_valid,n_reference_valid,cliffs_delta_comparison_vs_reference,p_raw
0,sex_age,Female − Male | age 60-69,SDNN,141,43,0.560943,2.687837e-08
1,sex_age,Female − Male | age 60-69,RMSSD,141,43,0.204024,4.323177e-02
2,sex_age,Female − Male | age 60-69,HF,141,43,0.206169,4.108238e-02
3,sex_age,Female − Male | age 60-69,LF,141,43,0.199406,4.819582e-02
4,sex_age,Female − Male | age 60-69,VLF,141,43,0.300016,2.942829e-03
5,sex_age,Female − Male | age 60-69,Mean RR,141,43,0.094838,3.478593e-01
6,sex_age,Female − Male | age 60-69,Mean HR,141,43,-0.090879,3.681339e-01
7,sex_age,Female − Male | age 70-79,SDNN,131,59,0.307672,7.023997e-04
8,sex_age,Female − Male | age 70-79,RMSSD,131,59,0.155259,8.739118e-02
9,sex_age,Female − Male | age 70-79,HF,131,59,0.132488,1.447294e-01


## 5. Multiple testing and reporting scope

Benjamini–Hochberg FDR is applied **separately within each of the six pair families**, and **separately for the five primary HRV and two complementary cardiac outcomes**. Each family includes all prespecified contrasts for the given pair, regardless of statistical significance. Undefined p-values from empty cells do not enter the denominator; their q-values remain undefined. Values are not adjusted jointly across all six pair families, across notebooks 06 or 07, or across the whole study.

This is a deliberately bounded **exploratory** multiplicity strategy analogous in transparency to stage 06. No significance-gated contrast generation is performed. The exact numerical p-values and q-values stay in CSV; reports display values below 0.001 as `p < 0.001` or `q < 0.001`.

In [6]:
effects["p_fdr_bh"] = np.nan
for (block,cls),indices in effects.groupby(["block","family"],sort=False).groups.items():
    effects.loc[indices,"p_fdr_bh"] = bh_fdr(effects.loc[indices,"p_raw"].to_numpy(dtype=float))
save_table(effects,"targeted_two_factor_effects.csv")

family_summary=[]
for block in BLOCK_ORDER:
    for family in ["primary_hrv","complementary_cardiac"]:
        part=effects[(effects.block==block)&(effects.family==family)]
        family_summary.append({
            "block":block,"pair":BLOCK_TITLES[block],"family":family,
            "n_preplanned_tests":len(part),"n_estimable_tests":int(part.p_raw.notna().sum()),
            "n_undefined_tests":int(part.p_raw.isna().sum()),
            "n_q_below_0_05":int((part.p_fdr_bh<.05).sum()),
            "min_group_n_observed":int(part[["n_comparison_cell","n_reference_cell"]].min().min()),
        })
family_summary=pd.DataFrame(family_summary)
save_table(family_summary,"two_factor_fdr_family_summary.csv")
display(family_summary)

,block,pair,family,n_preplanned_tests,n_estimable_tests,n_undefined_tests,n_q_below_0_05,min_group_n_observed
0,sex_age,Sex × age group,primary_hrv,25,25,0,5,6
1,sex_age,Sex × age group,complementary_cardiac,10,10,0,0,6
2,sex_sbp,Sex × systolic BP category,primary_hrv,20,20,0,4,31
3,sex_sbp,Sex × systolic BP category,complementary_cardiac,8,8,0,0,31
4,sex_dbp,Sex × diastolic BP category,primary_hrv,20,20,0,4,18
5,sex_dbp,Sex × diastolic BP category,complementary_cardiac,8,8,0,2,18
6,sbp_dbp,Systolic × diastolic BP categories,primary_hrv,25,25,0,0,15
7,sbp_dbp,Systolic × diastolic BP categories,complementary_cardiac,10,10,0,6,15
8,age_sbp,Age group × systolic BP category,primary_hrv,25,25,0,0,7
9,age_sbp,Age group × systolic BP category,complementary_cardiac,10,10,0,0,7


## 6. Visualization system

- **Cliff's-delta heatmap** for each pair: every predefined contrast × all seven outcomes, with the original comparison orientation and the within-pair BH significance marker. Undefined cells are labeled and never assigned a numeric color.
- **Primary-HRV forest plot** for each pair: five HRV panels with 95% percentile bootstrap intervals for Cliff's delta.
- **Complementary-cardiac forest plot** for each pair: Mean RR and Mean HR shown separately, reflecting their mathematical linkage.
- **Original-unit median-difference forest sheet** for each pair: seven panels of differences in measurement units with bootstrap intervals, separate from the rank-effect size.
- **Support overview** shows where comparisons rely on sparse or empty joint strata.

Color denotes within-pair adjusted significance for display only, not proof of clinical or mechanistic importance. Small cells are flagged with `*` and are never hidden. Text layout explicitly reserves vertical space for figure headings and panel titles.

In [7]:
figure_manifest=[]


def figure_header(fig,title,subtitle=""):
    h=fig.get_figheight()
    fig.suptitle(title,x=0.052,y=1-0.18/h,ha="left",va="top",
                 fontsize=17.5,color=NAVY,weight="bold")
    if subtitle:
        fig.text(0.052,1-.67/h,subtitle,ha="left",va="top",fontsize=9.6,color=MUTED)


def plot_area(fig,left=.16,right=.976,bottom_inches=.70,top_inches=1.42,wspace=.31,hspace=.48):
    h=fig.get_figheight()
    fig.subplots_adjust(left=left,right=right,bottom=bottom_inches/h,
                        top=1-top_inches/h,wspace=wspace,hspace=hspace)


def save_figure(fig,filename,block,role,description):
    target=fig_dir/filename
    fig.savefig(target,dpi=PLOT_DPI,bbox_inches="tight",pad_inches=.17)
    figure_manifest.append({"figure":str(target.relative_to(output_root)),
                            "block":block,"role":role,"description":description})
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)


def plot_support_overview():
    labels=[BLOCK_TITLES[b] for b in BLOCK_ORDER]
    x=np.arange(len(labels))
    group=ledger.groupby("block",sort=False)
    total=np.array([len(group.get_group(b)) for b in BLOCK_ORDER])
    empty=np.array([(group.get_group(b).n_comparison.eq(0)|group.get_group(b).n_reference.eq(0)).sum() for b in BLOCK_ORDER])
    small=np.array([((group.get_group(b)[["n_comparison","n_reference"]].min(axis=1)>0)&
                    (group.get_group(b)[["n_comparison","n_reference"]].min(axis=1)<10)).sum() for b in BLOCK_ORDER])
    fig,ax=plt.subplots(figsize=(13.2,7.2))
    ax.barh(x,total,color="#D8E9ED",height=.61,label="Planned contrast")
    ax.barh(x,small,color="#D8A06A",height=.61,label="Occupied group n<10")
    ax.barh(x,empty,color=CORAL,height=.61,label="Empty group (undefined)")
    ax.set_yticks(x,labels,fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel("Number of predefined comparisons")
    ax.set_xlim(0,max(total)+1.2)
    ax.legend(loc="lower right",frameon=False)
    ax.grid(axis="x")
    for i,n in enumerate(total):
        ax.text(n+.13,i,str(n),va="center",fontsize=9.5,color=NAVY)
    figure_header(fig,"Two-factor comparisons | Sample support",
                  "All planned contrasts retained | Small groups are precision warnings, not exclusions")
    plot_area(fig,left=.31,right=.96,top_inches=1.5,bottom_inches=.75)
    save_figure(fig,"00_targeted_contrast_support.png","all","support_overview",
                "Prespecified contrasts, sparse comparisons and undefined empty-cell comparisons")


def plot_cliff_heatmap(block,idx):
    sub=effects[effects.block==block]
    order=ledger.loc[ledger.block==block,"contrast_id"].tolist()
    values=sub.pivot(index="contrast_id",columns="outcome_label",values="cliffs_delta_comparison_vs_reference")
    values=values.reindex(index=order,columns=list(MEASURES.values()))
    qvals=sub.pivot(index="contrast_id",columns="outcome_label",values="p_fdr_bh").reindex(
        index=order,columns=list(MEASURES.values()))
    nmin=sub.drop_duplicates("contrast_id").set_index("contrast_id")[["n_comparison_cell","n_reference_cell"]].min(axis=1)
    ann=np.empty(values.shape,dtype=object)
    for i,cid in enumerate(values.index):
        for j,out in enumerate(values.columns):
            val=values.iloc[i,j]
            ann[i,j]=("not\nestimable" if not np.isfinite(val) else
                      f"{val:+.2f}" + (" †" if qvals.iloc[i,j]<ALPHA else "") +
                      (" *" if nmin.loc[cid]<10 else ""))
    fig,ax=plt.subplots(figsize=(15.4,6.0+0.60*len(order)))
    ax.set_facecolor("#EDF2F4")  # visible neutral background for undefined cells
    sns.heatmap(values,mask=values.isna(),cmap="RdBu_r",center=0,vmin=-.7,vmax=.7,
                linewidths=1.3,linecolor="white",annot=ann,fmt="",ax=ax,
                annot_kws={"fontsize":9.6},cbar_kws={"label":"Cliff's δ · comparison vs reference","shrink":.75})
    for i in range(len(order)):
        for j in range(len(MEASURES)):
            if not np.isfinite(values.iloc[i,j]):
                ax.text(j+.5,i+.5,"not\nestimable",ha="center",va="center",fontsize=9,color=MUTED)
    labels=ledger.set_index("contrast_id").loc[order,"contrast_label"].to_list()
    ax.set_yticks(np.arange(len(order))+.5,labels,rotation=0,fontsize=9.7)
    ax.set_xticks(np.arange(len(MEASURES))+.5,list(MEASURES.values()),rotation=0,fontsize=10)
    ax.set_xlabel("");ax.set_ylabel("")
    ax.tick_params(length=0)
    figure_header(fig,BLOCK_TITLES[block]+" | Cliff's delta atlas",
                  "Comparison minus reference | † within-pair BH q<0.05 | * occupied subgroup n<10 | Gray = undefined")
    plot_area(fig,left=.31,right=.95,top_inches=1.52,bottom_inches=.80)
    save_figure(fig,f"{idx:02d}_{block}_cliffs_delta_heatmap.png",block,"cliffs_delta_heatmap",
                "Seven-outcome conditional Cliff's delta heatmap with sample and FDR indicators")


def plot_delta_forest(block,idx,subset):
    outcomes=PRIMARY if subset=="primary" else COMPLEMENTARY
    rows=ledger[ledger.block==block].reset_index(drop=True)
    n=len(rows)
    width=20.6 if subset=="primary" else 13.4
    fig,axes=plt.subplots(1,len(outcomes),figsize=(width,5.1+0.49*n),sharey=True)
    axes=np.atleast_1d(axes)
    for panel,(ax,(outcome,title)) in enumerate(zip(axes,outcomes.items())):
        p=effects[(effects.block==block)&(effects.outcome==outcome)].set_index("contrast_id").loc[rows.contrast_id]
        for j,row in enumerate(p.itertuples(index=False)):
            y=n-1-j
            val=row.cliffs_delta_comparison_vs_reference
            if np.isfinite(val):
                significant=np.isfinite(row.p_fdr_bh) and row.p_fdr_bh<ALPHA
                clr=CORAL if significant else TEAL
                if np.isfinite(row.cliffs_ci_low) and np.isfinite(row.cliffs_ci_high):
                    ax.plot([row.cliffs_ci_low,row.cliffs_ci_high],[y,y],color=clr,linewidth=2.2,zorder=3)
                    ax.scatter([row.cliffs_ci_low,row.cliffs_ci_high],[y,y],marker="|",color=clr,s=75,zorder=4)
                marker="D" if min(row.n_comparison_valid,row.n_reference_valid)<10 else "o"
                ax.scatter(val,y,color=clr,s=38,marker=marker,zorder=5,edgecolor="white",linewidth=.45)
            else:
                ax.text(.0,y,"n=0",ha="center",va="center",fontsize=8,color=MUTED)
        ax.axvline(0,color=NAVY,linewidth=1.15,alpha=.72)
        ax.set_xlim(-1.04,1.04)
        ax.set_yticks(np.arange(n),rows.contrast_label.iloc[::-1].tolist())
        ax.tick_params(axis="y",labelsize=9.3,length=0)
        ax.tick_params(axis="x",labelsize=9.0)
        if panel>0:
            ax.tick_params(labelleft=False)
        ax.set_xlabel("Cliff's δ",fontsize=10)
        ax.set_title(title,pad=14,fontsize=12)
        ax.grid(axis="x",color="#E0EAF0",linewidth=.7)
        ax.set_axisbelow(True)
    figure_header(fig,BLOCK_TITLES[block]+f" | {('Primary HRV' if subset=='primary' else 'Complementary cardiac')} rank contrasts",
                  "Cliff's δ and bootstrap 95% CI | Orange: within-pair BH q<0.05 | Diamond: subgroup n<10 | Undefined: n=0")
    plot_area(fig,left=.26 if subset=="primary" else .36,right=.985,
              top_inches=1.48,bottom_inches=.72,wspace=.29)
    save_figure(fig,f"{idx:02d}_{block}_{subset}_cliffs_forest.png",block,subset+"_cliffs_forest",
                "Targeted two-factor Cliff's delta estimates and bootstrap confidence intervals")


def plot_median_difference_forest(block,idx):
    rows=ledger[ledger.block==block].reset_index(drop=True)
    fig,axes=plt.subplots(2,4,figsize=(20.0,12.7),sharey=True)
    axes=axes.flatten()
    for panel,(ax,(outcome,title)) in enumerate(zip(axes,MEASURES.items())):
        p=effects[(effects.block==block)&(effects.outcome==outcome)].set_index("contrast_id").loc[rows.contrast_id]
        all_values=np.r_[p.median_difference_comparison_minus_reference.to_numpy(dtype=float),
                         p.median_diff_ci_low.to_numpy(dtype=float),p.median_diff_ci_high.to_numpy(dtype=float)]
        finite=all_values[np.isfinite(all_values)]
        if len(finite):
            extent=max(abs(float(finite.min())),abs(float(finite.max())),1e-4)*1.22
            ax.set_xlim(-extent,extent)
        else:
            ax.set_xlim(-1,1)
        for j,row in enumerate(p.itertuples(index=False)):
            y=len(rows)-1-j
            val=row.median_difference_comparison_minus_reference
            if np.isfinite(val):
                clr=TEAL  # median-difference CI is not a Mann–Whitney significance statement
                if np.isfinite(row.median_diff_ci_low) and np.isfinite(row.median_diff_ci_high):
                    ax.plot([row.median_diff_ci_low,row.median_diff_ci_high],[y,y],color=clr,lw=2.1,zorder=3)
                    ax.scatter([row.median_diff_ci_low,row.median_diff_ci_high],[y,y],marker="|",s=65,color=clr,zorder=4)
                marker="D" if min(row.n_comparison_valid,row.n_reference_valid)<10 else "o"
                ax.scatter(val,y,marker=marker,s=31,color=clr,zorder=5)
            else:
                ax.text(0,y,"n=0",ha="center",va="center",fontsize=8,color=MUTED)
        ax.axvline(0,color=NAVY,lw=1.05,alpha=.72)
        ax.set_yticks(np.arange(len(rows)),rows.contrast_label.iloc[::-1].tolist())
        if panel % 4 != 0:
            ax.tick_params(labelleft=False)
        ax.tick_params(axis="y",labelsize=8.6,length=0)
        ax.tick_params(axis="x",labelsize=9)
        ax.set_title(title+" | median shift",pad=15,fontsize=11.6)
        ax.set_xlabel("Difference in source units",fontsize=9.7)
        ax.grid(axis="x",color="#E4ECF0",linewidth=.7)
        ax.set_axisbelow(True)
    axes[-1].set_visible(False)
    figure_header(fig,BLOCK_TITLES[block]+" | Original-unit median differences",
                  "Comparison minus reference | Percentile-bootstrap 95% CI | Distinct estimand from Mann–Whitney and Cliff's δ")
    plot_area(fig,left=.20,right=.975,bottom_inches=.85,top_inches=1.55,wspace=.42,hspace=.45)
    save_figure(fig,f"{idx:02d}_{block}_median_difference_forest.png",block,"median_difference_forest",
                "Median differences and bootstrap intervals in each measure's original source units")

plot_support_overview()
for idx,block in enumerate(BLOCK_ORDER,start=1):
    plot_cliff_heatmap(block,idx)
    plot_delta_forest(block,idx,"primary")
    plot_delta_forest(block,idx,"complementary")
    plot_median_difference_forest(block,idx)
print("Figures generated:",len(figure_manifest))

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.
findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Figures generated: 25


## 7. Export figures and reproducibility information

Every table and figure is written under the dedicated `07_02_targeted_two_factor_hrv_comparisons` directory. This step also generates a methodological README and a results summary containing the entire comparison ledger, family-level test counts, all effect estimates and their exact reporting direction. Numeric p-values and q-values are preserved in CSV; markdown uses `<0.001` formatting only for readability. Summaries do not promote significance as proof of mechanism, adjustment or interaction.

In [8]:
manifest=pd.DataFrame(figure_manifest)
save_table(manifest,"figure_manifest.csv")
run_information=pd.DataFrame([{
    "source_file":str(input_file),"n_records":len(data),
    "planned_contrasts":len(ledger),"occupied_cell_pair_contrasts":int(ledger.estimable_cell_pair.sum()),
    "not_estimable_empty_cell_contrasts":int((~ledger.estimable_cell_pair).sum()),
    "outcomes":len(MEASURES),"effects_rows":len(effects),"figures":len(manifest),
    "bootstrap_replicates":N_BOOT,"confidence_level":CONFIDENCE_LEVEL,"seed":SEED,
    "p_value_method":"Mann–Whitney asymptotic two-sided",
    "multiplicity":"BH within each pair family, separately primary HRV and complementary cardiac",
    "python":platform.python_version(),"pandas":pd.__version__,"numpy":np.__version__,
    "scipy":__import__('scipy').__version__,"seaborn":sns.__version__,"matplotlib":matplotlib.__version__,
}])
save_table(run_information,"run_information.csv")


def format_value(v,column):
    if pd.isna(v):return "NA"
    if isinstance(v,(bool,np.bool_)):return str(v)
    if isinstance(v,(int,np.integer)):return str(int(v))
    if isinstance(v,(float,np.floating)):
        if column in ("p_raw","p_fdr_bh"):
            return ("p < 0.001" if column=="p_raw" else "q < 0.001") if v<.001 else f"{v:.3f}"
        if column.startswith("n_"):return str(int(v))
        return f"{v:.3f}"
    return str(v).replace("|","/")


def md_table(frame,columns):
    a=frame.loc[:,columns].copy()
    for c in columns:a[c]=a[c].map(lambda v:format_value(v,c))
    return a.to_markdown(index=False)

readme=f"""# HRV analysis 07.02 — Targeted two-factor categorical comparisons

## Purpose and scope
Predefined, scientifically interpretable conditional comparisons of HRV and complementary cardiac measures in the accepted cohort, using existing sex, age-group, SBP-category and DBP-category variables. **This is not** a global factorial interaction model, a covariate-adjusted analysis, a causal analysis, a longitudinal comparison, or a significance-driven search through all possible two-factor cells. The complete descriptive atlas of all 10 categorical pairs belongs to 07.01; continuous-factor profiling and synthesis belong to 07.03.

## Dataset and outcome definition
- Input: `{input_file}`; records loaded: **{len(data)}**; source rows and original category thresholds retained.
- Primary HRV outcomes: {', '.join(PRIMARY.values())} (five separate measures).
- Complementary cardiac measures: {', '.join(COMPLEMENTARY.values())} (two separate but mathematically linked outcomes).
- No new exclusion, grouping, imputation, outcome replacement, source transformation or category merger is introduced.

## Six prespecified comparison families
{md_table(family_summary[["block","pair","family","n_preplanned_tests","n_estimable_tests","n_undefined_tests"]], ["pair","family","n_preplanned_tests","n_estimable_tests","n_undefined_tests"])}

The complete direction and each two-factor cell are documented in `tables/planned_two_factor_contrasts.csv`; it records the comparison and reference groups, sample sizes and status. A positive signed effect always refers to **comparison minus reference** (for Cliff's delta, more probable greater values in the comparison group). Within-group sex contrasts compare Female with Male; pressure contrasts compare high with low; age contrasts between 60–69 and 70–79 compare the older category with the younger.

The 80+ group is included in sex-within-age and pressure-within-age simple contrasts when a cell exists, with precision warnings for small cells. The **age 80+ × DBP ≥90** cell has no observations, so the corresponding comparison remains in tables with undefined effect size, interval, p and q. Sparse *occupied* groups are not silently dropped or labeled intrinsically non-estimable. Other age-category pairs were not added because they were outside the targeted central-age comparisons of the approved design.

## Statistical estimands
- Two-sided asymptotic Mann–Whitney U test: rank-distribution comparison; it is **not** automatically a test of medians.
- Cliff's δ: P(comparison > reference) − P(comparison < reference), ties counted as zero, in [-1,1].
- Sample median difference: median(comparison) − median(reference), in original measurement units.
- Stratified two-group **percentile bootstrap**: {N_BOOT} replicates, {CONFIDENCE_LEVEL:.0%} confidence intervals for δ and median differences, with resampling independently within each observed group; seed {SEED}.
- Full within-group descriptive profiles (n, mean, SD, P05/P25/median/P75/P95, IQR), including an explicit n=0 status for empty cells.
- No significance-gated comparisons, formal interaction contrasts or new adjusted regression models. Different p-values between strata do not establish effect heterogeneity.

## Multiple-testing scope
Benjamini–Hochberg FDR is applied **within each of the six two-factor pair families**, separately for the five main HRV measures and two complementary cardiac measures. Every estimable preplanned test in the corresponding family is included; undefined tests from empty cells are not counted and retain NA. FDR is **not pooled across the six pair families**, other stage 07 notebooks, stage 06 domains, or the full study. These q-values must never be reported as a global or study-wide FDR correction. Optional diagonal joint-BP comparison is included in its prespecified family and labeled as changing both factors.

The `p_raw` and `p_fdr_bh` CSV columns preserve numeric precision. In rendered Markdown, values below 0.001 appear as `p < 0.001` or `q < 0.001`. Effect estimates that round to 0.000 are numeric effect estimates, **not** p-values.

## Source preservation and uncertainty
All {len(data)} input records remain in the data frame. Group-specific comparisons naturally use their respective cells. CIs assume independent source records and **do not** account for clustering by source project. Empty cells cannot support a two-sample contrast; small occupied cells can yield very wide or unstable intervals. Differences are marginal within the specified two-factor stratum, but **not adjusted** for unconditioned covariates. The reported contrasts neither demonstrate causality nor formal sex-by-pressure or age-by-pressure interaction. HF, LF and VLF interpretation depends on the source protocol; Mean RR and Mean HR should not be read as independent confirmations.

## Generated files
All outputs are written to `eda_results/hrv_associations/07_02_targeted_two_factor_hrv_comparisons/`, never to upstream 05/06/07.01 folders.

- `tables/planned_two_factor_contrasts.csv` — exact contrast direction, two-factor cell identities, n and support status.
- `tables/targeted_two_factor_effects.csv` — all {len(effects)} outcome-contrast rows, effects, CIs, raw p and within-pair q, including undefined cells.
- `tables/two_factor_comparison_group_profiles.csv` — two descriptive cell profiles per planned contrast and outcome (including repeated cells with explicit provenance).
- `tables/two_factor_fdr_family_summary.csv` — per-family test denominators and undefined comparisons.
- `tables/figure_manifest.csv` — paths and content for every figure.
- `tables/run_information.csv` — source, environment and methodological configuration.
- `figures/multipanels/` — {len(manifest)} figures: one support overview and four per comparison family (effect heatmap; primary and complementary δ forests; original-unit median-difference forest).
- `README_07_02_targeted_two_factor_hrv_comparisons.md` — this methodological documentation.
- `07_02_targeted_two_factor_hrv_comparisons_results.md` — formatted findings table and complete contrast ledger.

## Execution
Run after accepted preprocessing and, for interpretation, after 07.01. No upstream output is needed to execute this notebook because all analyses use the accepted dataset. Use `INPUT_FILE_OVERRIDE`/`OUTPUT_ROOT_OVERRIDE` only if automatic path resolution is inappropriate. Set `SHOW_FIGURES=True` to display saved plots inline. Re-runs replace only their own stage-07.02 outputs.

## Environment
Python {platform.python_version()}, pandas {pd.__version__}, NumPy {np.__version__}, SciPy {__import__('scipy').__version__}, seaborn {sns.__version__}, Matplotlib {matplotlib.__version__}.
"""
(output_root/"README_07_02_targeted_two_factor_hrv_comparisons.md").write_text(readme,encoding="utf-8")

report=f"""# 07.02 — Targeted two-factor HRV comparisons | Results

**Records:** {len(data)} | **Prespecified contrasts:** {len(ledger)} | **Estimable pair contrasts:** {int(ledger.estimable_cell_pair.sum())} | **Bootstrap:** {N_BOOT}, {CONFIDENCE_LEVEL:.0%} CI

## Interpretation boundary
Predefined conditional comparisons across existing two-factor categories, **not** factorial interaction models, causality or covariate-adjusted effects. Cliff's δ and median difference have different meanings. Sample precision varies across contrasts, especially for the 80+ strata. Mean RR/HR are complementary and mathematically related. Numeric p/q values remain in CSV. BH-FDR is applied within pair families separately for HRV and complementary cardiac outcomes, **not globally across stage 07 or the project**.

## Complete comparison ledger and support
{md_table(ledger,["contrast_id","block","contrast_label","n_comparison","n_reference","support_status"])}

## Multiple-testing families
{md_table(family_summary,["pair","family","n_preplanned_tests","n_estimable_tests","n_undefined_tests","n_q_below_0_05"])}

## Complete effect-size results
All planned comparisons are included, including non-significant effects and empty cells. Positive Cliff's delta describes higher stochastic ordering in the comparison group; positive median difference describes higher observed sample medians. A difference in significance between strata is not evidence that the underlying effects differ.

{md_table(effects,["block_label","contrast_label","outcome_label","n_comparison_valid","n_reference_valid","cliffs_delta_comparison_vs_reference","cliffs_ci_low","cliffs_ci_high","median_difference_comparison_minus_reference","median_diff_ci_low","median_diff_ci_high","p_raw","p_fdr_bh"])}

## Figure index
{md_table(manifest,["figure","description"])}

## Limitations
This is an unadjusted cross-sectional analysis. All source rows are retained; only observed members of predefined cells contribute to a corresponding contrast. Small groups have low precision, and an empty category cell has undefined two-sample effects. Independent-record bootstrap intervals do not account for possible source-project clustering. No evidence of formal interaction, causal effects, or medical diagnostic categories is established by these contrasts.
"""
(output_root/"07_02_targeted_two_factor_hrv_comparisons_results.md").write_text(report,encoding="utf-8")

print("Output:",output_root)
print("Tables:",len(list(tables_dir.glob('*.csv'))),"| Figures:",len(manifest),
      "| README + report generated")

Output: /home/david/Desktop/colabs/nunez_cristian_colabs/heart_variability_rate_profiles/eda_results/hrv_associations/07_02_targeted_two_factor_hrv_comparisons
Tables: 6 | Figures: 25 | README + report generated


## 8. Stage completion checklist

- Every prespecified two-factor contrast and each of the seven cardiac measurements is represented, **including empty-cell rows**.
- The complete comparison orientation, actual n and descriptive support warning are recorded for every estimate.
- Raw statistical outputs are kept numerically precise; markdown is a presentation layer.
- The 06/07.01 input files remain unchanged, and the 07.02 results are written to their own directory.
- The final decision about subsequent conditional continuous analyses belongs to 07.03.

**Validation note:** Run this notebook with the accepted real cohort and review the exported tables/figures before treating the estimates as scientific findings; a synthetic-data smoke test only verifies software execution.